In [1]:
CONFIG = {
  #================================#
  #=== CONFIGURAÇÕES DO DATASET ===#
  #================================#
  "dataset": "../data/ETTh2.csv",

  #===============================#
  #=== CONJUNTO DE TREINAMENTO ===#
  #===============================#
  "train_start": "2016-07-12 06:00:00",
  "train_end": "2016-09-20 05:00:00",

  #===============================#
  #=== CONFIGURAÇÕES DO MODELO ===#
  #===============================#
  "model": "amazon/chronos-2",
  "device_map": "cpu",
  "forecast_horizon": 24,

  #==============#
  #=== OUTPUT ===#
  #==============#
  "folder": "experiments/ETTh2_Chronos2"
}

In [2]:
import llm4series as ls
import pandas as pd
import plotly.express as px
from scipy.stats import sem
import time
import os

import logging
logging.getLogger("apscheduler").setLevel(logging.CRITICAL)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)
logging.getLogger("requests").setLevel(logging.CRITICAL)

import warnings
warnings.simplefilter("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# === CarbonTracker ===
import io
import re
from contextlib import redirect_stdout
from carbontracker.tracker import CarbonTracker as Tracker

# === Eco2AI ===
import eco2ai
eco2ai_tracker = eco2ai.Tracker(project_name="eco2ai", file_name="eco2ai_emissions.csv",alpha_2_code="BR", ignore_warnings=True)

In [3]:
class CarbonTracker:
  def __init__(self, epochs=1, verbose=1):
    self.epochs = epochs
    self.verbose = verbose
    self.buffer = io.StringIO()
    self.tracker = None
    self.co2_g = None
    self.energy_kwh = None

  def __enter__(self):
    self._stdout_ctx = redirect_stdout(self.buffer)
    self._stdout_ctx.__enter__()
    self.tracker = Tracker(epochs=self.epochs, verbose=self.verbose)
    self.tracker.epoch_start()
    return self

  def __exit__(self, exc_type, exc, tb):
    self.tracker.epoch_end()
    self.tracker.stop()
    self._stdout_ctx.__exit__(exc_type, exc, tb)
    output = self.buffer.getvalue()

    match_co2 = re.search(r"CO2eq:\s*([\d\.]+)\s*g", output)
    self.co2_g = float(match_co2.group(1)) if match_co2 else None

    match_energy = re.search(r"Energy:\s*([\d\.]+)\s*kWh", output)
    self.energy_kwh = float(match_energy.group(1)) if match_energy else None

In [4]:
ts = ls.read_file(CONFIG["dataset"])
freq = "d"
print(f"✅ Total de períodos: {len(ts)}")
print(f"📅 Frequência: {freq}")

ts_train, ts_val = ts.split(start=CONFIG["train_start"], end=CONFIG["train_end"], periods=CONFIG["forecast_horizon"])
print(f"✅ Total de períodos de treino: {len(ts_train)}")
print(f"✅ Total de períodos de validação: {len(ts_val)}")

import torch
from chronos import Chronos2Pipeline
model = Chronos2Pipeline.from_pretrained(CONFIG["model"], device_map=CONFIG["device_map"], torch_dtype=torch.float32)
print(f"✅ Modelo: {CONFIG['model']}")

✅ Total de períodos: 1704
📅 Frequência: d
✅ Total de períodos de treino: 1680
✅ Total de períodos de validação: 24


/home/liaan/Documentos/neurips/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


✅ Modelo: amazon/chronos-2


In [5]:
import time

train_df = pd.DataFrame({
  "id": "s1",
  "timestamp": ts_train.index,
  "target": ts_train.values
})

start_time = time.time()
eco2ai_tracker.start()
with CarbonTracker() as carbontracker:
  pred_df = model.predict_df(
    train_df,
    prediction_length=CONFIG["forecast_horizon"],
    quantile_levels=[0.1, 0.5, 0.9],
    id_column="id",
    timestamp_column="timestamp",
    target="target",
  )
end_time = time.time()

carbontracker_g = carbontracker.co2_g
carbontracker_kWh = carbontracker.energy_kwh
carbontracker_Wh = carbontracker_kWh * 1000

eco2ai_tracker.stop()
eco2ai_data = pd.read_csv("eco2ai_emissions.csv").iloc[-1]
eco2ai_kWh = eco2ai_data["power_consumption(kWh)"]
eco2ai_Wh = eco2ai_kWh * 1000
eco2ai_kg = eco2ai_data["CO2_emissions(kg)"]
eco2ai_g = eco2ai_kg * 1000

pred = pred_df["predictions"].tolist()
metrics = ls.metrics(pred, ts_val)["value"]

results = [{
  "Dataset": CONFIG["dataset"],
  "Início do Treinamento": CONFIG["train_start"],
  "Fim do Treinamento": CONFIG["train_start"],
  "Horizonte de Previsão": CONFIG["forecast_horizon"],
  "Modelo": CONFIG["model"],
  "sMAPE": round(metrics["smape"], 4),
  "MAE": round(metrics["mae"], 4),
  "RMSE": round(metrics["rmse"], 4),
  "CarbonTracker CO₂ (g)": round(carbontracker_g, 6),
  "CarbonTracker Consumo (Wh)": round(carbontracker_Wh, 6),
  "Eco2AI CO₂ (g)": round(eco2ai_g, 6),
  "Eco2AI Consumo (Wh)": round(eco2ai_Wh, 6),
  "Tempo de Inferência (s)": round(end_time - start_time, 4),
  "Valores Reais": list(ts_val),
  "Valores Preditos": list(pred),
}]

[ERROR] Status code 429 from http://ipinfo.io/json: ERROR - 429 Client Error: Too Many Requests for url: http://ipinfo.io/json


In [6]:
df = pd.DataFrame(results)
os.makedirs(CONFIG["folder"], exist_ok=True)
df.to_csv(f"{CONFIG["folder"]}/resultado_experimentos.csv", index=False)
print(f"✅ Resultados salvos: {len(df)} experimentos")
df

✅ Resultados salvos: 1 experimentos


,Dataset,Início do Treinamento,Fim do Treinamento,Horizonte de Previsão,Modelo,sMAPE,MAE,RMSE,CarbonTracker CO₂ (g),CarbonTracker Consumo (Wh),Eco2AI CO₂ (g),Eco2AI Consumo (Wh),Tempo de Inferência (s),Valores Reais,Valores Preditos
0,../data/ETTh2.csv,2016-07-12 06:00:00,2016-07-12 06:00:00,24,amazon/chronos-2,5.13,1.78,2.27,0.001213,0.002726,0.001181,0.010448,1.4822,"[25.438815843419672, 25.11408604030149, 27.441...","[25.591323852539062, 25.876052856445312, 26.57..."


In [7]:
df = pd.read_csv(f"{CONFIG["folder"]}/resultado_experimentos.csv")

row = df.iloc[0]
smape = row["sMAPE"]
mae = row["MAE"]
rmse = row["RMSE"]

ts_val = ls.UniTimeSeries(eval(row["Valores Reais"]))
ts_pred = ls.UniTimeSeries(eval(row["Valores Preditos"]))
ls.lineplot(ts_val, ts_pred, title=f"sMAPE = {smape} / MAE = {mae} / RMSE = {rmse}", groups=["Real", "Predicted"])